# Image GPS Matching

This notebook matches each image/annotation with its corresponding GPS data based on the Cityscapes `image_id`.

For each image and annotation (e.g. `"aachen_000000_000019_leftImg8bit.png"` and `"aachen_000000_000019_gtFine_color.png"`), 
there exists a corresponding JSON file `"aachen_000000_000019_vehicle.json"` containing GPS metadata.

The `match_images_with_gps()` function takes as arguments:
- The path to the root directory of the Cityscapes dataset
- The annotation type (`"gtFine"` or `"gtCoarse"`) to set the correct directories for annotations and JSON files

And returns a DataFrame with columns `["split", "city", "image_id", "latitude", "longitude"]` for each image/annotation.

In [ ]:
import json
from tqdm import tqdm
import pandas as pd
from pathlib import Path

def match_images_with_gps(data_path, annotation_type):
    """
    Matches Cityscapes images with their corresponding GPS coordinates. 
    
    Pairing each image with GPS metadata from its corresponding vehicle JSON file based on matching image IDs.
    """

     # Set directory paths based on annotation type
    if annotation_type == "gtFine":
        annot_path = data_path / "gtFine_trainvaltest" / annotation_type
        json_path = data_path / "vehicle_trainvaltest/vehicle"

        # Fine annotations have train, val, and test splits
        split_paths = list(annot_path.glob("*"))
        splits = [split_path.name for split_path in split_paths]

    elif annotation_type == "gtCoarse":
        annot_path = data_path /"gtCoarse" / annotation_type
        json_path = data_path / "vehicle_trainextra/vehicle"

        # Coarse annotations only have train_extra split
        splits = ["train_extra"]
    else:
        print('Invalid annotation type provided. Valid types are "gtFine" or "gtCoarse".')
        return None

    result_rows = []

    # Iterate through all splits
    for split in splits:
        # Iterate through all cities in the split
        city_paths = list((annot_path / split).glob("*"))
        for city_path in city_paths:
            city = city_path.name
            
            # Get all annotation files for this city
            img_paths = list(city_path.glob(f"*_{annotation_type}_color.png"))
        
            for img_path in tqdm(img_paths, total=len(img_paths), desc=f"{annotation_type}: Matching images in {city}"):
                # Extract image_id by removing the annotation type suffix
                image_id = img_path.name.replace(f"_{annotation_type}_color.png", "")

                # Read corresponding GPS data from vehicle JSON file
                json_file = json_path / split / city / f"{image_id}_vehicle.json"
                
                if not json_file.exists():
                    print(f"JSON-File at {json_file} does not exist")
                    continue
                
                with open(json_file, "r") as file:
                    data = json.load(file)
                
                 # Add result row with image_id and GPS data
                result_rows.append({
                    "split": split,
                    "city": city,
                    "image_id": image_id,
                    "latitude": data["gpsLatitude"],
                    "longitude": data["gpsLongitude"]
                })

    return pd.DataFrame(result_rows)

## Process Both Cityscapes Variants

Apply GPS matching to both fine and coarse Cityscapes datasets (`"gtFine"` and `"gtCoarse"`).


In [21]:
data_path = Path("../data/City Scapes")

cityscapes_fine_gps_data = match_images_with_gps(data_path, "gtFine")
cityscapes_fine_gps_data.head()

gtFine: Matching images in munster: 100%|██████████| 174/174 [00:00<00:00, 7310.74it/s]


,split,city,image_id,latitude,longitude
0,test,berlin,berlin_000000_000019,52.497751,13.295325
1,test,berlin,berlin_000001_000019,52.498672,13.300393
2,test,berlin,berlin_000002_000019,52.499890,13.306958
3,test,berlin,berlin_000003_000019,52.499929,13.307324
4,test,berlin,berlin_000004_000019,52.498154,13.306086


In [22]:
cityscapes_coarse_gps_data = match_images_with_gps(data_path, "gtCoarse")
cityscapes_coarse_gps_data.head()

gtCoarse: Matching images in troisdorf:  71%|███████   | 98/138 [00:00<00:00, 151.28it/s]

JSON-File at ..\data\City Scapes\vehicle_trainextra\vehicle\train_extra\troisdorf\troisdorf_000000_000073_vehicle.json does not exist


gtCoarse: Matching images in wurzburg: 100%|██████████| 1622/1622 [00:11<00:00, 145.36it/s]


,split,city,image_id,latitude,longitude
0,train_extra,augsburg,augsburg_000000_000000,48.371323,10.886690
1,train_extra,augsburg,augsburg_000000_000001,48.371254,10.887023
2,train_extra,augsburg,augsburg_000000_000002,48.371205,10.887244
3,train_extra,augsburg,augsburg_000000_000003,48.371137,10.887553
4,train_extra,augsburg,augsburg_000000_000004,48.371093,10.887759


## Export GPS-Matched DataFrames

Export the `image_id` and corresponding GPS coordinates as CSV files

In [23]:
cityscapes_fine_gps_data.to_csv("../output/cityscapes_fine_gps_data.csv")
cityscapes_coarse_gps_data.to_csv("../output/cityscapes_coarse_gps_data.csv")